# LangChain Foundations — LLM → Memory → RAG → Agents

## Setup & VS Code notes
1. Create venv (if not already):
```bash
python -m venv ollama-env
```
2. Activate venv:
- Windows: `ollama-env\Scripts\activate`
- Mac/Linux: `source ollama-env/bin/activate`

3. In VS Code: `Ctrl+Shift+P` → `Python: Select Interpreter` → pick `ollama-env`.
4. Install packages (run the cell below). Jupyter inside VS Code will use the selected interpreter/venv.

> **Note:** This notebook targets Python 3.10+. The code avoids syntax requiring newer versions. If running on a different 3.x, it should still work.

## LangChain — LLM Wrapper (Why we use it)
- LangChain provides a consistent interface for many LLMs (OpenAI, Ollama, Bedrock, HF, local models).
- You get prompt templates, chains, memory helpers, tools, and agents with consistent APIs.

#Connecting LangChain with Ollama

LangChain connects to different Large Language Models (LLMs) through wrappers.
These wrappers make it easy to send a prompt and receive a model response
without handling HTTP requests manually.

For local models, **Ollama** is the easiest option.  
It allows running open-source models like *Mistral, Llama 3, Phi-3, Gemma,* etc. directly on your system.

LangChain’s **`Ollama`** class (from `langchain_community.llms`) provides a simple API
to send prompts to any Ollama model.

---

### ⚙️ Basic Flow
1. Ollama service runs locally at `http://localhost:11434`
2. LangChain’s `Ollama` wrapper connects to it
3. You call `.invoke(prompt)` or `.generate([...])` to get model output

---

### ✨ Key Advantages
- No API keys or cloud costs
- Works offline
- Fast for prototyping and teaching


In [ ]:
# Example 1: Test LangChain <-> Ollama connection
from langchain_community.llms import Ollama

# Initialize the local LLM connection
# This will connect to the Ollama daemon running on localhost
llm = Ollama(model="mistral")

# Send a simple prompt
response = llm.invoke("Say a one-line hello message from Mistral!")

print(response)

In [ ]:
# Example 2: Setting optional parameters
llm = Ollama(model="mistral", temperature=0.7)

prompt = "List three advantages of using containerization in IT infrastructure."
print(llm.invoke(prompt))

In [ ]:
# Example 3: Using .generate() for multiple prompts
prompts = [
    "Explain virtualization in one line.",
    "What is DevOps?",
    "What are APIs in simple terms?"
]

result = llm.generate(prompts)

# .generate() returns an object with generations for each prompt
for i, gen in enumerate(result.generations):
    print(f"Prompt {i+1}: {prompts[i]}")
    print("Response:", gen[0].text, "\n")

In [ ]:
print(result.generations)
for i, gen in enumerate(result.generations):
    print(f"Prompt {i+1}: {prompts[i]}")
    print("Response:", gen[0].text, "\n")

In [ ]:
# Example 4: Streaming output (useful for chat UIs)
for chunk in llm.stream("Explain microservices architecture in short."):
    print(chunk, end="", flush=True)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
#from langchain_community.llms import Ollama

In [ ]:
prompt.format

In [ ]:
# Initialize the model
llm = Ollama(model="mistral")

# Create a simple template
template = "Explain {topic} in simple terms for a new IT trainee."

# Define the prompt template with the variable 'topic'
prompt = PromptTemplate(template=template, input_variables=["topic"])

# Fill the template with a topic
final_prompt = prompt.format(topic="cloud computing")

print("Generated Prompt:\n", final_prompt)

# Invoke the model
response = llm.invoke(final_prompt)
print("\nModel Response:\n", response)

In [ ]:
# Example 2: Using multiple variables
# Template with two variables
template = """
You are an IT trainer. 
Explain the concept of {technology} and give one real-world {application_area} example.
"""
# Define prompt
prompt = PromptTemplate(
    template=template,
    input_variables=["technology", "application_area"]
)
# Fill the variables dynamically
filled_prompt = prompt.format(
    technology="containerization",
    application_area="DevOps"
)
print("Generated Prompt:\n", filled_prompt)
# Send to model
response = llm.invoke(filled_prompt)
print("\nModel Response:\n", response)

In [ ]:
# Example 3: Function for reusable prompt generation
def explain_it_concept(topic, example_domain):
    template = """
    You are an IT mentor.
    Explain the concept of {topic} in 4-5 lines with a relevant example from {example_domain}.
    """
    prompt = PromptTemplate(template=template, input_variables=["topic", "example_domain"])
    final_prompt = prompt.format(topic=topic, example_domain=example_domain)
    return llm.invoke(final_prompt)

# Test the function
print(explain_it_concept("microservices", "banking systems"))

In [ ]:
help(ChatPromptTemplate)

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel

class KingInfo(BaseModel):
    name: str
    reign_period: str

parser = PydanticOutputParser(pydantic_object=KingInfo)

# Example usage (pseudo): instruct the model to return JSON and feed to parser
print('Use parser.parse(model_output) in exercises to validate structure.')

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

# Automatically validates and converts data
user = User(name="Alice", age="30")  # string "30" becomes int 30
print(user.age)  # 30

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define structure of expected output
class KingInfo(BaseModel):
    name: str = Field(..., description="Name of the king")
    country: str = Field(..., description="Country they ruled")

# Create parser
parser = PydanticOutputParser(pydantic_object=KingInfo)
print(parser.get_format_instructions())

# Example model output
model_output = """
{
    "name": "King Arthur",
    "country": "apple"
}
"""
# Parse and validate
parsed = parser.parse(model_output)
print(parsed)
# KingInfo(name='King Arthur', country='Britain')
print(parsed.dict())
# {'name': 'King Arthur', 'country': 'Britain'}

In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(..., min_length=2)
    age: int = Field(..., gt=0, lt=150)


In [ ]:
from pydantic import BaseModel, Field
from enum import Enum

class Country(str, Enum):
    britain = "Britain"
    france = "France"
    egypt = "Egypt"

class KingInfo(BaseModel):
    name: str
    country: Country

LangChain historically offered many special-purpose **chains** (SQL, API, Router, Summarize, etc.).  
As the ecosystem matured, these have consolidated into three modern layers:

- **Runnables** — compose logic flexibly (the new foundation)
- **RAG** — retrieval-based tasks (production knowledge assistants)
- **Agents** — dynamic decision-making + tool use

We will still learn the **core chains** because they teach the foundations of orchestration.  
Everything else you’ll recognize conceptually (for legacy code), but you won’t need to implement.

---
## ✅ Learn & Implement (core)
- **LLMChain** — prompt + model (one step)
- **SimpleSequentialChain** — single input → multi-step → single output
- **SequentialChain** — multi-input/multi-output pipelines
- **Summarization chains**  
  - **Stuff** (small inputs)  
  - **Map-Reduce** (scales across chunks)  
  - **Refine** (iterative, detail-preserving)

## ⚙️ Know Conceptually (legacy/common, replaced by modern patterns)
- **Retrieval/QA chains**: `RetrievalQA`, `ConversationalRetrievalChain`  
  → *Replaced by* **RAG** (retriever + prompt + LLM via Runnables)
- **Routing/selection**: `RouterChain`, `MultiPromptChain`, `LLMRouterChain`  
  → *Replaced by* **LangGraph routers** / **Supervisor agents**
- **Data & API**: `SQLDatabaseChain`, `APIChain`, `PAL`  
  → *Replaced by* **Agents + Tools** (SQL tools, API tools, code tools)
- **Guardrails/policy**: `ConstitutionalChain`  
  → *Replaced by* **output parsers/validators** and hosted guardrails
- **Glue/observability**: `TransformChain`, `AnalyzeDocumentChain`  
  → *Replaced by* **RunnableLambda/Map** and tracing

# 🔗 Topic 3: Workflows with Runnables (Modern Replacement for Chains)

LangChain used to use “Chains” like `LLMChain`, `SequentialChain`, and `MapReduceChain` to connect prompts and models.  
In 2025+, these have been replaced by **Runnables** — a lighter, composable abstraction that does the same job more flexibly.

---

## ⚙️ What Are Runnables?

A **Runnable** is any object that:
- Takes input data
- Processes it (prompt formatting, logic, model call)
- Produces output

You can combine Runnables using the `|` (pipe) operator, just like UNIX pipes.  
This gives you the same pipeline behavior as classic Chains but works with the latest LangChain versions.

---

### 🧩 Common Runnable Types

| Runnable | Purpose | Example |
|-----------|----------|---------|
| `RunnableLambda` | Wraps a custom Python function | Transform or preprocess data |
| `RunnablePassthrough` | Passes data through unchanged | Use as pipeline start |
| `RunnableMap` | Run multiple branches in parallel | e.g., summary + quiz |
| `RunnableSequence` | Combine multiple runnables sequentially | multi-step workflow |

---

### 🧱 Modern Replacements

| Classic Chain | Modern Equivalent |
|----------------|------------------|
| `LLMChain` | `RunnablePassthrough() | PromptTemplate | LLM` |
| `SequentialChain` | `RunnableSequence` |
| `SimpleSequentialChain` | Chained `|` pipes |
| `MapReduceChain` | `RunnableMap` + summarizer |

---

### ✅ Why This Matters

- Fully compatible with **LangChain 0.4+**
- More modular and debuggable
- Standard for **RAG** and **Agents** internally

In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = Ollama(model="mistral")

prompt = PromptTemplate(
    template="Explain the concept of {topic} in 3 lines suitable for a new IT employee.",
    input_variables=["topic"]
)

# Runnable pipeline: Input → Prompt → LLM
#chain = RunnablePassthrough() | (lambda x: prompt.format(**x)) | llm
chain = RunnablePassthrough() | prompt | llm
response = chain.invoke({"topic": "DevOps"})
print(response)

In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 1️⃣ Create model (Ollama + mistral)
model = Ollama(model="mistral")

# 2️⃣ Create parser
parser = CommaSeparatedListOutputParser()

# 3️⃣ Create prompt
prompt = PromptTemplate.from_template(
    "List three fruits {format_instructions}"
)

# 4️⃣ Build Runnable chain
chain = prompt | model | parser

# 5️⃣ Run it
result = chain.invoke({"format_instructions": parser.get_format_instructions()})
print(result)
# Example: ['apple', 'banana', 'cherry']

In [ ]:
parser.get_format_instructions()

In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1️⃣ Define structured schema
class KingInfo(BaseModel):
    name: str = Field(..., description="Name of the king")
    country: str = Field(..., description="Country they ruled")

# 2️⃣ Create parser
parser = PydanticOutputParser(pydantic_object=KingInfo)

# 3️⃣ Create model (Ollama mistral)
model = Ollama(model="mistral")

# 4️⃣ Create prompt template
prompt = PromptTemplate.from_template("""
Extract information about the king from this text:
{text}

{format_instructions}
""")

# 5️⃣ Build Runnable chain
chain = prompt | model | parser

# 6️⃣ Invoke it
text = "King Arthur was a legendary British leader from Camelot."
result = chain.invoke({
    "text": text,
    "format_instructions": parser.get_format_instructions()
})

print(result)
# KingInfo(name='King Arthur', country='Britain')

print(result.dict())
# {'name': 'King Arthur', 'country': 'Britain'}

In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.exceptions import OutputParserException
from pydantic import BaseModel, Field
from enum import Enum

# 1️⃣ Define strict schema with Enum for validation
class Country(str, Enum):
    britain = "Britain"
    france = "France"
    egypt = "Egypt"

class KingInfo(BaseModel):
    name: str = Field(..., description="Name of the king")
    country: Country = Field(..., description="Country they ruled")

parser = PydanticOutputParser(pydantic_object=KingInfo)

# 2️⃣ Create model
model = Ollama(model="mistral")

# 3️⃣ Create prompt template
prompt = PromptTemplate.from_template("""
Extract information about the king from this text:
{text}

{format_instructions}
""")

# 4️⃣ Build the base chain
base_chain = prompt | model | parser

# 5️⃣ Retry wrapper
def safe_invoke(inputs):
    try:
        return base_chain.invoke(inputs)
    except OutputParserException as e:
        # Retry by adding clarification to the prompt
        print("⚠️ Parsing failed, retrying...")
        inputs["text"] += (
            "\n\nPlease reformat your answer as valid JSON and choose a valid country "
            "from: Britain, France, or Egypt."
        )
        return base_chain.invoke(inputs)

# Wrap in a RunnableLambda for composability
safe_chain = RunnablePassthrough() | RunnableLambda(safe_invoke)

# 6️⃣ Run it
text = "King Arthur was a mythical British leader of Camelot, not France or Egypt."
result = safe_chain.invoke({
    "text": text,
    "format_instructions": parser.get_format_instructions()
})

print(result)
print(result.dict())

In [ ]:
from langchain_core.runnables import RunnableSequence
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
llm = Ollama(model="mistral")

# Step 1 – Summarize
prompt1 = PromptTemplate(
    template="Summarize {topic} in one line.",
    input_variables=["topic"]
)

# Step 2 – Give an IT example
prompt2 = PromptTemplate(
    template="Give one real-world IT example for this summary: {summary}",
    input_variables=["summary"]
)

# Step 3 – Build sequential workflow
def summarize(inputs):
    return {"summary": llm.invoke(prompt1.format(**inputs))}

def example(inputs):
    return {"example": llm.invoke(prompt2.format(**inputs))}

#workflow = RunnableSequence(first=RunnableLambda(summarize), middle=[], last=RunnableLambda(example))
ex = RunnableSequence()
result = workflow.invoke({"topic": "microservices architecture"})
print(result)


In [ ]:
from langchain_core.runnables import RunnableMap

llm = Ollama(model="mistral")

prompt_summary = PromptTemplate.from_template("Summarize {topic} in 2 lines.")
prompt_question = PromptTemplate.from_template("Create one interview question on {topic}.")

parallel_chain = RunnableMap({
    "summary": prompt_summary | llm,
    "question": prompt_question | llm
})

result = parallel_chain.invoke({"topic": "virtualization"})
print(result)

# 🧩 Topic 4: Memory in LangChain (Modern Runnable-Compatible Approach)

Memory lets a conversation-based app **remember context across turns**.  
Without it, the model would treat every prompt as independent.

---

### 🧱 Why Memory Matters
| Without Memory | With Memory |
|----------------|-------------|
| “Who are you?” → “I’m Mistral.”<br>“What’s my name?” → “I don’t know.” | “Who are you?” → “I’m Mistral.”<br>“What’s my name?” → “You’re Mistral.” |

---

### 🧩 Types of Memory (Conceptually)
| Memory Type | What It Stores | Typical Use |
|--------------|---------------|--------------|
| **ConversationBufferMemory** | Full conversation transcript | Short chats |
| **ConversationSummaryMemory** | Summarized conversation | Long-running sessions |
| **ConversationTokenBufferMemory** | Recent tokens only | Context window management |
| **EntityMemory** | Facts about entities (people, places, etc.) | Personalized chatbots |

---

### ⚙️ How Memory Works Now (2025+)
Earlier versions used `ConversationChain(memory=...)`.  
Now we can plug memories directly into **Runnables** or **Agent Executors** by manually managing the stored messages.

In other words:  
> “Memory is just structured context we append to prompts before calling the LLM.”

In [ ]:
from langchain_community.llms import Ollama
from langchain_community.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = Ollama(model="mistral")
memory = ConversationBufferMemory(return_messages=True)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful IT assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

def conversation_step(user_input):
    history = memory.load_memory_variables({})["history"]
    formatted = prompt.format_messages(history=history, input=user_input)
    reply = llm.invoke(formatted)
    memory.save_context({"input": user_input}, {"output": reply})
    return reply

print(conversation_step("Hi, who are you?"))
print(conversation_step("Can you remind me what I just asked?"))


Buffer Memory with Runnables

In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough

llm = Ollama(model="mistral")
history = []

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful IT support assistant."),
    MessagesPlaceholder("history"),
    ("human", "{input}")
])

def memory_step(user_input):
    """Simulate buffer memory manually."""
    history.append(("human", user_input))
    messages = prompt.format_messages(history=history, input=user_input)
    reply = llm.invoke(messages)
    history.append(("assistant", reply))
    return reply

print(memory_step("Hi, who are you?"))
print(memory_step("Can you tell me what I just asked?"))

Summarized (Condensed) Memory using Runnables

In [ ]:
from langchain_core.runnables import RunnableLambda

summary = None

def summarize_history(history):
    """Summarize long conversation using LLM."""
    text = "\n".join([f"{r}: {m}" for r, m in history])
    return llm.invoke(f"Summarize the following conversation in 4-5 lines:\n{text}")

def chat_with_summary(user_input):
    global summary

    # If there’s a summary, include it before the chat
    system_prompt = "You are a concise IT assistant."
    history_msgs = history.copy()
    if summary:
        history_msgs.insert(0, ("system", f"Summary so far: {summary}"))

    history_msgs.append(("human", user_input))
    messages = prompt.format_messages(history=history_msgs, input=user_input)
    reply = llm.invoke(messages)
    history.append(("assistant", reply))

    # Summarize if conversation gets too long
    if len(history) > 8:
        summary = summarize_history(history)
        history.clear()
        history.append(("system", f"Condensed summary: {summary}"))

    return reply

print(chat_with_summary("What is cloud computing?"))
print(chat_with_summary("Explain how it's storage differs from local storage."))


In [ ]:
summary = None

def hybrid_chat(user_input):
    global summary
    # keep only last 3 exchanges
    short_history = history[-6:]

    combined = []
    if summary:
        combined.append(("system", f"Earlier summary: {summary}"))
    combined += short_history
    combined.append(("human", user_input))

    messages = prompt.format_messages(history=combined, input=user_input)
    reply = llm.invoke(messages)
    history.append(("assistant", reply))

    # update summary every 6 messages
    if len(history) % 6 == 0:
        summary = summarize_history(history)

    return reply

print(hybrid_chat("Tell me about virtualization."))
print(hybrid_chat("How does it compare to containerization?"))


In [ ]:
import json
print(json.dumps(history, indent=2))
print("Summary:", summary)


In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda

llm = Ollama(model="mistral")
history = []

# summarizer Runnable
summarizer = RunnableLambda(lambda x: llm.invoke(f"Summarize this chat:\n{x}"))

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise IT assistant."),
    MessagesPlaceholder("history"),
    ("human", "{input}")
])

def chat_step(user_input):
    # append new turn
    history.append(("human", user_input))
    formatted = prompt.format_messages(history=history, input=user_input)
    reply = llm.invoke(formatted)
    history.append(("assistant", reply))

    # if history too long → condense
    if len(history) > 10:
        summary_text = summarizer.invoke(str(history))
        history.clear()
        history.append(("system", f"Summary so far: {summary_text}"))

    return reply

print(chat_step("Explain virtualization."))
print(chat_step("And how is it different from containerization?"))


In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import json, os

# ------------------------------
# 1. Load / Save History (same)
# ------------------------------
HISTORY_FILE = "faq_chat.json"

def load_chat():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, "r", encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                return []
    return []

def save_chat(messages):
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(messages, f, indent=2)

# -----------------------------------
# 2. Define LLM + Prompt (Runnable)
# -----------------------------------
llm = Ollama(model="mistral")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful IT FAQ assistant."),
    MessagesPlaceholder("history"),
    ("human", "{input}")
])

# -----------------------------------
# 3. Chat loop using Runnable logic
# -----------------------------------
history = load_chat()

def chat_step(user_input):
    # Prepare message history for LangChain
    formatted_messages = [(m["role"], m["content"]) for m in history]
    formatted_messages.append(("human", user_input))

    # Format + run through LLM
    messages = prompt.format_messages(history=formatted_messages, input=user_input)
    reply = llm.invoke(messages)

    # Update and persist
    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": reply})
    save_chat(history)
    return reply

# -----------------------------------
# 4. Main Interaction
# -----------------------------------
print("💻 IT FAQ Assistant (Runnable version)\nType 'exit' to quit.\n")
while True:
    user = input("You: ").strip()
    if user.lower() == "exit":
        save_chat(history)
        break
    print("Assistant:", chat_step(user))
